# Item-Based Collaborative Filtering

## Introduction

Item-Based Collaborative Filtering is a recommendation technique that predicts a user's preference for an item based on the similarity between items. Unlike User-Based Collaborative Filtering, which finds similar users, Item-Based Collaborative Filtering focuses on identifying relationships between items. This approach is particularly effective when the number of users is large but the number of items is relatively stable.

The key idea is simple: **if a user liked certain items in the past, then they are likely to enjoy similar items in the future.** Therefore, recommendations are generated by analyzing how items relate to each other based on historical user interactions. This method is widely used in real-world systems because it is computationally efficient, scalable, and often produces stable recommendations.

---

### Step 1 — Construct the User–Item Interaction Matrix

The first step in Item-Based Collaborative Filtering is constructing a **User–Item Matrix**. This matrix represents user interactions with items, such as ratings, purchases, or clicks.

Let:

* $U = {u_1, u_2, ..., u_m}$ be the set of users
* $I = {i_1, i_2, ..., i_n}$ be the set of items

Step 1 in **item‑based collaborative filtering** is about turning your raw user–item data into a clean, numerical matrix that you can later use to compute how similar items are to each other. Below is a more explicit, math‑heavy explanation of this step.

#### 1. What is the goal of Step 1?
Construct a **user–item interaction matrix** $R \in \mathbb{R}^{m \times n}$, where:
- Each **row** corresponds to a user $u \in U = \{u_1, u_2,\dots,u_m\}$.
- Each **column** corresponds to an item $i \in I = \{i_1, i_2,\dots,i_n\}$.
- Element $R_{u,i}$ stores the interaction between user $u$ and item $i$ (rating, purchase, click, etc.), or 0 if there is no interaction.

#### 2. Formal mathematical definition
Let:
- $U = \{u_1, u_2, \dots, u_m\}$ be the set of $m$ users.
- $I = \{i_1, i_2, \dots, i_n\}$ be the set of $n$ items.

Then the **interaction matrix** is
$$
R = \begin{bmatrix}
R_{u_1,i_1} & R_{u_1,i_2} & \cdots & R_{u_1,i_n} \\
R_{u_2,i_1} & R_{u_2,i_2} & \cdots & R_{u_2,i_n} \\
\vdots      & \vdots      & \ddots & \vdots  \\
R_{u_m,i_1} & R_{u_m,i_2} & \cdots & R_{u_m,i_n}
\end{bmatrix}
\in \mathbb{R}^{m \times n}.
$$

Each entry is defined as
$$
R_{u,i} =
\begin{cases}
r_{u,i} \in \mathbb{R} & \text{if user } u \text{ interacted with item } i, \\
0 & \text{if there is no interaction for } (u,i).
\end{cases}
$$

Here $r_{u,i}$ can be:
- an explicit rating (e.g., 1–5 stars), or  
- an implicit signal (e.g., 1 = clicked, 0 = no click; or 5 = number of views, 0 = no view).

#### 3. Concrete example with indices
Take your example table:
| User | Item A | Item B | Item C | Item D |
|------|--------|--------|--------|--------|
| U1   | 5      | 3      | 0      | 1      |
| U2   | 4      | 0      | 0      | 1      |
| U3   | 1      | 1      | 0      | 5      |
| U4   | 0      | 0      | 5      | 4      |

Assign indices:
- Users:  
  - $u_1 = \text{U1},\ u_2 = \text{U2},\ u_3 = \text{U3},\ u_4 = \text{U4}$.  
- Items:  
  - $i_1 = \text{Item A},\ i_2 = \text{Item B},\ i_3 = \text{Item C},\ i_4 = \text{Item D}$.  

Then the matrix $R$ becomes:
$$
R =
\begin{bmatrix}
5 & 3 & 0 & 1 \\
4 & 0 & 0 & 1 \\
1 & 1 & 0 & 5 \\
0 & 0 & 5 & 4
\end{bmatrix}
\in \mathbb{R}^{4 \times 4}.
$$

So:
- $R_{u_1, i_1} = R_{1,1} = 5$ $\rightarrow$ user U1 gave rating 5 to Item A.
- $R_{u_2, i_2} = R_{2,2} = 0$ $\rightarrow$ user U2 has no interaction with Item B (unrated/unseen).

#### 4. Why this matrix is "the foundation"
- Each **row** $R_{u,*}$ is a vector of length $n$ that encodes all interactions of user $u$ with all items.
- Each **column** $R_{*,i}$ is a vector of length $m$ that encodes how all users interacted with item $i$.

In **item‑based filtering**, you will later:
- Treat each **column** as an "item vector."
- Compute **similarity between columns** (e.g., using cosine similarity) to decide which items are "alike."

So Step 1 is: **map your raw data into this matrix $R$**, and only then move to Step 2 (item‑item similarity computation).

#### 5. How to implement this step (pseudocode style)
Given raw data like:
$$
\text{data} = \{(u, i, r_{u,i})\}
$$

where each triple is (user, item, rating), you:
1. Map users to indices:
   - `user2idx = {u1:0, u2:1, u3:2, u4:3}`
2. Map items to indices:
   - `item2idx = {A:0, B:1, C:2, D:3}`
3. Initialize a zero matrix $R \in \mathbb{R}^{m \times n}$.
4. For each triple $(u, i, r)$:
   - $R[\text{user2idx}[u], \ \text{item2idx}[i]] = r$.

After this, you have the exact matrix in the example above, ready for item‑based similarity calculations.

---

### Step 2 — Compute Item Similarity

After constructing the interaction matrix, the next step is computing **item-to-item similarity**.

Common similarity measures include:

* Cosine Similarity
* Pearson Correlation
* Adjusted Cosine Similarity

#### Cosine Similarity

Cosine similarity measures the angle between two item vectors.

For two items $i$ and $j$:

$$
\text{sim}(i,j) =
\frac{
\sum_{u \in U} R_{u,i} \cdot R_{u,j}
}{
\sqrt{\sum_{u \in U} R_{u,i}^2}
\cdot
\sqrt{\sum_{u \in U} R_{u,j}^2}
}
$$

Where:

* $R_{u,i}$ is the rating of user $u$ for item $i$
* $R_{u,j}$ is the rating of user $u$ for item $j$

The similarity value ranges between:

$$
-1 \leq \text{sim}(i,j) \leq 1
$$

Higher values indicate stronger similarity between items.

---

### Step 3 — Build Item Similarity Matrix

After computing similarity between every pair of items, we construct the **Item Similarity Matrix**:

$$
S \in \mathbb{R}^{n \times n}
$$

where:

$$
S_{i,j} = \text{sim}(i,j)
$$

Example:

| Item | A   | B   | C   | D   |
| ---- | --- | --- | --- | --- |
| A    | 1   | 0.8 | 0.1 | 0.4 |
| B    | 0.8 | 1   | 0.2 | 0.3 |
| C    | 0.1 | 0.2 | 1   | 0.9 |
| D    | 0.4 | 0.3 | 0.9 | 1   |

This matrix tells us which items are most similar to each other.

---

### Step 4 — Predict User Preference

To recommend items, we estimate how much a user would like an item they have not interacted with.

The predicted score is computed using weighted similarity:

$$
\hat{R}*{u,i} =
\frac{
\sum*{j \in N(i)} \text{sim}(i,j) \cdot R_{u,j}
}{
\sum_{j \in N(i)} |\text{sim}(i,j)|
}
$$

Where:

* $\hat{R}_{u,i}$ = predicted rating
* $N(i)$ = set of similar items to item $i$
* $R_{u,j}$ = user rating for item $j$

This formula computes a **weighted average** of ratings based on item similarity.

---

### Step 5 — Generate Top-N Recommendations

After computing predicted scores, we generate recommendations:

1. Predict scores for all unseen items
2. Sort items by predicted score
3. Select Top-N highest scoring items

Formally:

$$
\text{Top-N}(u) =
\underset{i \notin I_u}{\operatorname{arg,max}} \ \hat{R}_{u,i}
$$

Where:

* $I_u$ = items already interacted by user $u$
* $\hat{R}_{u,i}$ = predicted score

---

### Step 6 — Advantages of Item-Based Filtering

Item-Based Collaborative Filtering has several advantages:

* More scalable for large user bases
* Stable similarity between items
* Efficient for real-time recommendation
* Works well with sparse datasets

---

### Step 7 — Limitations

However, this approach also has limitations:

* Cold start problem for new items
* Requires sufficient interaction data
* Similarity computation may be expensive for very large item sets

---

### Summary

Item-Based Collaborative Filtering follows these steps:

1. Build User–Item Matrix
2. Compute Item Similarity
3. Construct Similarity Matrix
4. Predict User Preference
5. Generate Top-N Recommendations

This method is widely used in recommender systems such as:

* E-commerce product recommendations
* Movie recommendation systems
* Music recommendation platforms

Because it balances **accuracy**, **scalability**, and **interpretability**, Item-Based Collaborative Filtering remains one of the most fundamental approaches in recommender systems.


## Module Loading

In [1]:
!pip install -q duckdb polars joblib tqdm matplotlib seaborn scipy mermaid-py

In [2]:
import os
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
from pylab import rcParams
import pandas as pd
import numpy as np
from google.colab import drive
from warnings import filterwarnings

darkmodel = False
rcParams['figure.figsize'] = (12,6)
pd.options.display.float_format = '{:,.2f}'.format
filterwarnings('ignore', category=FutureWarning)

%matplotlib inline

if darkmodel:
    # 1. Define a sophisticated E-commerce color palette
    # These colors are chosen for high contrast against the #212946 background
    colors = [
        "#08F7FE",  # Cyan Glow
        "#FE53BB",  # Neon Pink
        "#F5D300",  # Cyber Yellow
        "#00ff41",  # Matrix Green
        "#9467bd",  # Royal Purple
    ]

    # 2. Enhanced Dictionary with Complex Styling
    refined_dark_style = {
        # Background and Canvas
        "figure.facecolor": "#212946",
        "axes.facecolor": "#212946",
        "savefig.facecolor": "#212946",

        # Grid Sophistication
        "axes.grid": True,
        "axes.grid.which": "both",
        "grid.color": "#2A3459",
        "grid.linewidth": "1",
        "grid.alpha": 0.5,

        # Typography & Labels (Optimized for readability)
        "text.color": "#E2E2E2",
        "axes.labelcolor": "#E2E2E2",
        "axes.labelsize": 14,
        "axes.titlesize": 18,
        "axes.titleweight": "bold",
        "axes.titlepad": 20,
        "xtick.color": "#8E9CC3",
        "ytick.color": "#8E9CC3",
        "font.size": 12,

        # Spines (Clean aesthetic)
        "axes.spines.left": False,
        "axes.spines.right": False,
        "axes.spines.top": False,
        "axes.spines.bottom": True,
        "axes.edgecolor": "#2A3459",

        # Line & Marker Settings
        "lines.linewidth": 2.5,
        "lines.markersize": 8,
        "axes.prop_cycle": plt.cycler(color=colors),
    }

    plt.rcParams.update(refined_dark_style)

else:
    # 1. Defining the "Paper & Ink" Palette
    # Deep Blue #003366 | Oxide Red #A52A2A
    ecom_vintage_colors = [
        "#003366",  # Oxford Blue (Primary)
        "#A52A2A",  # Oxide Red (Comparison)
        "#006400",  # Dark Green (Success Metrics)
        "#704214",  # Sepia (Neutral)
    ]

    vintage_style = {
        # Background - The specific parchment hex you requested
        "figure.facecolor": "#f7e4b7",
        "axes.facecolor": "#f7e4b7",
        "savefig.facecolor": "#f7e4b7",

        # Grid - Subtle contrast using a darker version of the background
        "axes.grid": True,
        "grid.color": "#e2d1a8",
        "grid.linestyle": "-",
        "grid.linewidth": 1.0,

        # Typography - Deep Charcoal/Blue instead of pure black for a softer feel
        "text.color": "#2C2C2C",
        "axes.labelcolor": "#2C2C2C",
        "xtick.color": "#5D5D5D",
        "ytick.color": "#5D5D5D",
        "axes.titlesize": 16,
        "axes.titleweight": "bold",
        "axes.titlepad": 15,
        "font.size": 11,

        # Spines - Classic 'L-frame' for publication
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.spines.left": True,
        "axes.spines.bottom": True,
        "axes.edgecolor": "#2C2C2C",
        "axes.linewidth": 1.5,

        # Data Point Styling
        "axes.prop_cycle": plt.cycler(color=ecom_vintage_colors),
        "lines.linewidth": 2.2,
        "lines.markersize": 8,
        "patch.edgecolor": "#f7e4b7", # Borders on bars/pie slices
    }

    plt.rcParams.update(vintage_style)

In [3]:
# ======================================================
# Logging Configuration
# ======================================================
import logging
logger = logging.getLogger(__name__)
logger.setLevel(logging.INFO)

if not logger.handlers:
    handler = logging.StreamHandler()
    formatter = logging.Formatter(
        "%(asctime)s | %(levelname)s | %(name)s | %(message)s"
    )
    handler.setFormatter(formatter)
    logger.addHandler(handler)


## Data Loading

In [4]:
drive.mount('/content/drive')
pd.set_option('display.max_rows', 50)

def ListFiles(Dirs):
    errormsg = f"Error: Directory '{Dirs}' does not exist or is not a directory."
    assert os.path.isdir(Dirs), errormsg
    file_data = list()
    for item in os.listdir(Dirs):
        item_path = os.path.join(Dirs, item)
        if os.path.isfile(item_path):
            try:
                size_bytes = os.path.getsize(item_path)
                size_mb = size_bytes / (1024 * 1024)  # Convert bytes to MB
                file_data.append({'File Name': item, 'Size (MB)': size_mb})
            except Exception as err:
                print(f"Could not get size for {item_path}: {err}")

    Files = pd.DataFrame(file_data)
    return Files

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
MyFiles = ListFiles('/content/drive/MyDrive/Colab Notebooks')
DatFilename = MyFiles[~MyFiles['File Name'].str.contains('.ipynb', na = False)]
display(DatFilename)

,File Name,Size (MB)
76,MasterData.parquet,145.58
79,InstaCart.db,263.76
82,xgboost_ltr_model.json,2.43
83,catboost_ltr_model.cbm,0.32
84,lightgbm_ltr_model.txt,6.21
85,lgbm_optuna_ranker_model.txt,0.58
86,test_df.parquet,3.84


In [6]:
import duckdb as dc

DBfilepath = '/content/drive/MyDrive/Colab Notebooks/' + 'InstaCart.db'
try:
    dbcon = dc.connect(DBfilepath)
    chk = dbcon.execute("PRAGMA show_tables;").fetchdf()
except dc.DuckDBError as e:
    print(f"An DuckDB error occurred: {e}")
finally:
    display(chk)
    #if 'dbcon' in locals() and dbcon is not None:
    #    dbcon.close()

,name
0,AISLE
1,DepartmentData
2,FullTrainData
3,OrderTest
4,OrderTrain
5,OrdersDetails
6,ProductsData


In [7]:
masterdata = dbcon.execute("SELECT * FROM FullTrainData;").fetchdf()
masterdata['order_hour_of_day'] = masterdata['order_hour_of_day'].astype('category')
display(masterdata.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1384617 entries, 0 to 1384616
Data columns (total 11 columns):
 #   Column                  Non-Null Count    Dtype   
---  ------                  --------------    -----   
 0   order_id                1384617 non-null  int64   
 1   user_id                 1384617 non-null  int64   
 2   product_id              1384617 non-null  int64   
 3   aisle_id                1384617 non-null  int64   
 4   department_id           1384617 non-null  int64   
 5   order_number            1384617 non-null  int64   
 6   order_dow               1384617 non-null  int64   
 7   order_hour_of_day       1384617 non-null  category
 8   days_since_prior_order  1384617 non-null  float64 
 9   add_to_cart_order       1384617 non-null  int64   
 10  reordered               1384617 non-null  int64   
dtypes: category(1), float64(1), int64(9)
memory usage: 107.0 MB


None

In [8]:
import pandas as pd
import numpy as np
import duckdb as dc
import polars as pl
from tqdm.auto import tqdm

class FeatureEngineer:

    # ======================================================
    # Initialization
    # ======================================================
    def __init__(self, data: pd.DataFrame):

        if not isinstance(data, pd.DataFrame):
            raise TypeError(
                f"Input must be pandas DataFrame, got {type(data)}"
            )

        # Normalize headers
        self.Data = data.copy()
        self.Data.columns = [
            col.strip().lower() for col in self.Data.columns
        ]

        required_cols = [
            'user_id',
            'product_id',
            'order_number',
            'reordered'
        ]

        missing = [
            col for col in required_cols
            if col not in self.Data.columns
        ]

        if missing:
            raise KeyError(
                f"Critical columns missing: {missing}"
            )

        self.user_stats = None
        self.product_stats = None
        self.df_feat = None
        self.feature_cols = None

        logger.info(
            f"FeatureEngineer Initialized | Rows: {len(self.Data)} | Cols: {len(self.Data.columns)}"
        )


    # ======================================================
    # User Feature Engineering
    # ======================================================
    def create_user_features(self):

        logger.debug("Creating user features...")

        agg_map = {
            'order_number': 'max',
            'days_since_prior_order': 'mean',
            'add_to_cart_order': 'mean',
            'reordered': 'sum',
        }

        available_aggs = {
            k: v for k, v in agg_map.items()
            if k in self.Data.columns
        }

        try:

            self.user_stats = (
                self.Data
                .groupby('user_id')
                .agg(available_aggs)
                .reset_index()
            )

            rename_map = {
                'order_number': 'user_total_orders',
                'days_since_prior_order': 'user_avg_days_between',
                'add_to_cart_order': 'user_avg_cart_pos',
                'reordered': 'user_total_reorders'
            }

            self.user_stats.rename(
                columns=rename_map,
                inplace=True
            )

            if self.user_stats.empty:
                raise ValueError(
                    "User feature generation produced empty dataframe"
                )

        except Exception:
            logger.exception("Error creating user features")
            raise


    # ======================================================
    # Product Feature Engineering
    # ======================================================
    def create_product_features(self):

        logger.debug("Creating product features...")

        try:

            stats = self.Data.groupby('product_id').agg({
                'reordered': ['sum', 'mean'],
                'order_id': 'count' if 'order_id' in self.Data.columns else 'size',
                'add_to_cart_order': 'mean'
            })

            stats.columns = [
                'prod_total_reorders',
                'prod_reorder_rate',
                'prod_order_count',
                'prod_avg_cart_pos'
            ]

            self.product_stats = stats.reset_index()

            nan_count = (
                self.product_stats['prod_reorder_rate']
                .isnull()
                .sum()
            )

            if nan_count > 0:
                logger.debug(
                    f"{nan_count} products contain NaN reorder rate"
                )

        except Exception:
            logger.exception("Error creating product features")
            raise


    # ======================================================
    # Merge Engines
    # ======================================================
    def merge_features_pandas(self):

        logger.debug("Merging using Pandas...")

        self.df_feat = self.Data.merge(
            self.user_stats,
            on='user_id',
            how='left'
        )

        self.df_feat = self.df_feat.merge(
            self.product_stats,
            on='product_id',
            how='left'
        )


    def merge_features_duckdb(self):

        logger.debug("Merging using DuckDB...")

        con = dc.connect(database=':memory:')

        self_Data = self.Data
        self_user_stats = self.user_stats
        self_product_stats = self.product_stats

        query = """
        SELECT
            d.*,
            u.* EXCLUDE (user_id),
            p.* EXCLUDE (product_id)
        FROM self_Data as d
        LEFT JOIN self_user_stats as u
        ON d.user_id = u.user_id
        LEFT JOIN self_product_stats as p
        ON d.product_id = p.product_id
        """

        self.df_feat = con.execute(query).df()


    def merge_features_polars(self):

        logger.debug("Merging using Polars...")

        ldf = pl.from_pandas(self.Data)
        u_stats = pl.from_pandas(self.user_stats)
        p_stats = pl.from_pandas(self.product_stats)

        try:

            self.df_feat = (
                ldf
                .join(
                    u_stats,
                    on="user_id",
                    how="left",
                    validate="m:1"
                )
                .join(
                    p_stats,
                    on="product_id",
                    how="left",
                    validate="m:1"
                )
            ).to_pandas()

        except Exception:
            logger.exception("Polars merge failed")
            raise


    # ======================================================
    # Smart Merge Selection
    # ======================================================
    def merge_features(self):

        if self.user_stats is None or self.product_stats is None:
            raise ValueError(
                "Features must be computed before merge"
            )

        original_count = len(self.Data)

        if original_count <= 50_000:
            self.merge_features_pandas()

        elif original_count <= 700_000:
            self.merge_features_polars()

        else:
            self.merge_features_duckdb()

        if len(self.df_feat) > original_count:
            logger.warning(
                f"Row explosion detected {original_count} -> {len(self.df_feat)}"
            )


    # ======================================================
    # Cleanup Garbage Columns
    # ======================================================
    def _cleanup_columns(self):

        bad_cols = [
            'index',
            'level_0',
            'unnamed: 0'
        ]

        drop_cols = [
            c for c in bad_cols
            if c in self.df_feat.columns
        ]

        if drop_cols:
            logger.debug(
                f"Dropping unnecessary columns: {drop_cols}"
            )

            self.df_feat.drop(
                columns=drop_cols,
                inplace=True
            )


    # ======================================================
    # Fit Transform
    # ======================================================
    def fit_transform(self, show_progress=True):

        steps = [
            ("Create User Features", self.create_user_features),
            ("Create Product Features", self.create_product_features),
            ("Merge Features", self.merge_features),
            ("Finalize Features", None)
        ]

        try:

            iterator = steps

            if show_progress:
                iterator = tqdm(
                    steps,
                    desc="Feature Engineering",
                    leave=True,
                    dynamic_ncols=True
                )

            for step_name, step_func in iterator:

                logger.debug(f"Running Step: {step_name}")

                if step_func:
                    step_func()

                else:

                    potential_features = [
                        'order_number', 'order_dow', 'order_hour_of_day',
                        'days_since_prior_order', 'add_to_cart_order',
                        'user_total_orders', 'user_avg_days_between',
                        'user_avg_cart_pos', 'user_total_reorders',
                        'prod_total_reorders', 'prod_reorder_rate',
                        'prod_order_count', 'prod_avg_cart_pos'
                    ]

                    self.feature_cols = [
                        c for c in potential_features
                        if c in self.df_feat.columns
                    ]

                    numeric_feats = (
                        self.df_feat[self.feature_cols]
                        .select_dtypes(include=[np.number])
                        .columns
                    )

                    medians = self.df_feat[numeric_feats].median()

                    self.df_feat.fillna(
                        medians,
                        inplace=True
                    )

            self._cleanup_columns()

            logger.info(
                f"fit_transform completed | Features: {len(self.feature_cols)}"
            )

            return (
                self.df_feat,
                numeric_feats,
                self.feature_cols
            )

        except Exception:
            logger.exception(
                "CRITICAL ERROR in fit_transform"
            )

            return None, None, None


In [9]:
FEs = FeatureEngineer(data = masterdata)
masterdata2, feature_cols, others = FEs.fit_transform()

2026-04-07 04:02:26,288 | INFO | __main__ | FeatureEngineer Initialized | Rows: 1384617 | Cols: 11
INFO:__main__:FeatureEngineer Initialized | Rows: 1384617 | Cols: 11


Feature Engineering:   0%|          | 0/4 [00:00<?, ?it/s]

2026-04-07 04:02:29,593 | INFO | __main__ | fit_transform completed | Features: 13
INFO:__main__:fit_transform completed | Features: 13


In [10]:
display(masterdata2.info())
display(masterdata2.sample(5))

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1384617 entries, 0 to 1384616
Data columns (total 19 columns):
 #   Column                  Non-Null Count    Dtype   
---  ------                  --------------    -----   
 0   order_id                1384617 non-null  int64   
 1   user_id                 1384617 non-null  int64   
 2   product_id              1384617 non-null  int64   
 3   aisle_id                1384617 non-null  int64   
 4   department_id           1384617 non-null  int64   
 5   order_number            1384617 non-null  int64   
 6   order_dow               1384617 non-null  int64   
 7   order_hour_of_day       1384617 non-null  category
 8   days_since_prior_order  1384617 non-null  float64 
 9   add_to_cart_order       1384617 non-null  int64   
 10  reordered               1384617 non-null  int64   
 11  user_total_orders       1384617 non-null  int64   
 12  user_avg_days_between   1384617 non-null  float64 
 13  user_avg_cart_pos       1384617 non-null  

None

,order_id,user_id,product_id,aisle_id,department_id,order_number,order_dow,order_hour_of_day,days_since_prior_order,add_to_cart_order,reordered,user_total_orders,user_avg_days_between,user_avg_cart_pos,user_total_reorders,prod_total_reorders,prod_reorder_rate,prod_order_count,prod_avg_cart_pos
1321580,3188924,160666,4605,83,4,10,4,10,30.00,5,1,10,30.00,7.00,3,2548,0.68,3762,8.42
262890,592081,161877,17948,116,1,16,1,15,17.00,1,1,16,17.00,11.00,14,688,0.65,1065,9.91
959775,2206670,54221,6884,57,14,7,3,08,17.00,1,1,7,17.00,1.00,1,8,0.47,17,7.00
1375429,3265551,141795,30406,83,4,5,2,22,30.00,4,0,5,30.00,3.00,3,187,0.44,421,8.53
1344673,3400126,205754,30563,100,21,51,1,09,7.00,10,1,51,7.00,12.50,24,63,0.49,129,9.59


## Step 1. Build User–Item Matrix

In [11]:
masterdata2.shape

(1384617, 19)

In [12]:
import gc
import numpy as np
import pandas as pd

from scipy.sparse import csr_matrix, vstack
from joblib import Parallel, delayed
from tqdm.auto import tqdm

from contextlib import contextmanager
import joblib

In [13]:
@contextmanager
def tqdm_joblib(tqdm_object):
    class TqdmBatchCompletionCallback(joblib.parallel.BatchCompletionCallBack):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = TqdmBatchCompletionCallback

    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_callback
        tqdm_object.close()


In [14]:
class ItemBasedCFBuilder:
    """
    Memory-Efficient User-Item Matrix Builder
    Production-Scale Friendly
    """

    def __init__(
        self,
        user_col: str = "user_id",
        item_col: str = "product_id",
        value_col: str = "reordered",
        n_jobs: int = -1
    ):

        self.user_col = user_col
        self.item_col = item_col
        self.value_col = value_col
        self.n_jobs = n_jobs

        self.user_mapping = None
        self.item_mapping = None

        self.user_inverse_mapping = None
        self.item_inverse_mapping = None


    # --------------------------------------------------
    # Step 1 — Mapping
    # --------------------------------------------------

    def create_id_mappings(self, df: pd.DataFrame):

        logger.info("Creating user/item mappings...")

        users = df[self.user_col].unique()
        items = df[self.item_col].unique()

        self.user_mapping = {u: i for i, u in enumerate(users)}
        self.item_mapping = {i: j for j, i in enumerate(items)}

        self.user_inverse_mapping = {v: k for k, v in self.user_mapping.items()}
        self.item_inverse_mapping = {v: k for k, v in self.item_mapping.items()}

        logger.info(
            f"Users: {len(users)} | Items: {len(items)}"
        )


    # --------------------------------------------------
    # Step 2 — Chunk Generator (Memory Safe)
    # --------------------------------------------------

    def dataframe_chunks(
        self,
        df: pd.DataFrame,
        chunk_size: int
    ):

        for start in range(0, len(df), chunk_size):
            yield df.iloc[start:start + chunk_size]


    # --------------------------------------------------
    # Step 3 — Vectorized Chunk Processing
    # --------------------------------------------------

    def process_chunk(self, chunk: pd.DataFrame):

        logger.debug(f"Processing chunk size: {len(chunk)}")

        user_idx = chunk[self.user_col].map(self.user_mapping).values
        item_idx = chunk[self.item_col].map(self.item_mapping).values
        values = chunk[self.value_col].values

        n_users = len(self.user_mapping)
        n_items = len(self.item_mapping)

        sparse_chunk = csr_matrix(
            (values, (user_idx, item_idx)),
            shape=(n_users, n_items)
        )

        del chunk
        gc.collect()

        return sparse_chunk


    # --------------------------------------------------
    # Step 4 — Parallel Processing (Streaming)
    # --------------------------------------------------

    def parallel_build_matrix(
        self,
        df: pd.DataFrame,
        chunk_size: int
    ):

        logger.info("Building sparse matrix in parallel...")

        chunks = list(
            self.dataframe_chunks(
                df,
                chunk_size
            )
        )

        matrices = []

        with tqdm_joblib(
            tqdm(
                total=len(chunks),
                desc="Building Sparse Matrix",
                colour="green"
            )
        ):

            matrices = Parallel(
                n_jobs=self.n_jobs,
                backend="loky"
            )(
                delayed(self.process_chunk)(chunk)
                for chunk in chunks
            )

        logger.info("Parallel processing completed")

        logger.info("Stacking sparse matrices...")

        final_matrix = vstack(matrices)

        del matrices
        gc.collect()

        return final_matrix


    # --------------------------------------------------
    # Main Pipeline
    # --------------------------------------------------

    def build_user_item_matrix(
        self,
        df: pd.DataFrame,
        chunk_size: int = 200_000
    ):

        logger.info("Starting User-Item Matrix pipeline...")

        self.create_id_mappings(df)

        matrix = self.parallel_build_matrix(
            df,
            chunk_size
        )

        logger.info(
            f"Matrix shape: {matrix.shape}"
        )

        logger.info(
            f"Non-zero elements: {matrix.nnz}"
        )

        sparsity = 1 - matrix.nnz / (
            matrix.shape[0] * matrix.shape[1]
        )

        logger.info(
            f"Sparsity: {sparsity:.6f}"
        )

        logger.info("User-Item Matrix Completed")

        return matrix

In [15]:
builder = ItemBasedCFBuilder(user_col="user_id",
                             item_col="product_id",
                             value_col="reordered",
                             n_jobs=-1,
                             )

user_item_matrix = builder.build_user_item_matrix(
    masterdata2,
    chunk_size=200_000
)

2026-04-07 04:02:30,365 | INFO | __main__ | Starting User-Item Matrix pipeline...
INFO:__main__:Starting User-Item Matrix pipeline...
2026-04-07 04:02:30,374 | INFO | __main__ | Creating user/item mappings...
INFO:__main__:Creating user/item mappings...
2026-04-07 04:02:30,649 | INFO | __main__ | Users: 131209 | Items: 39123
INFO:__main__:Users: 131209 | Items: 39123
2026-04-07 04:02:30,657 | INFO | __main__ | Building sparse matrix in parallel...
INFO:__main__:Building sparse matrix in parallel...


Building Sparse Matrix:   0%|          | 0/7 [00:00<?, ?it/s]

2026-04-07 04:02:51,629 | INFO | __main__ | Parallel processing completed
INFO:__main__:Parallel processing completed
2026-04-07 04:02:51,631 | INFO | __main__ | Stacking sparse matrices...
INFO:__main__:Stacking sparse matrices...
2026-04-07 04:02:51,828 | INFO | __main__ | Matrix shape: (918463, 39123)
INFO:__main__:Matrix shape: (918463, 39123)
2026-04-07 04:02:51,830 | INFO | __main__ | Non-zero elements: 1384617
INFO:__main__:Non-zero elements: 1384617
2026-04-07 04:02:51,831 | INFO | __main__ | Sparsity: 0.999961
INFO:__main__:Sparsity: 0.999961
2026-04-07 04:02:51,832 | INFO | __main__ | User-Item Matrix Completed
INFO:__main__:User-Item Matrix Completed


In [16]:
user_item_matrix.shape

(918463, 39123)

## Step 2. Compute Item Similarity

In [17]:
import gc
import joblib
import numpy as np
from tqdm.auto import tqdm
from scipy.sparse import csr_matrix
from sklearn.metrics.pairwise import cosine_similarity
from joblib import Parallel, delayed
from contextlib import contextmanager

In [18]:
@contextmanager
def tqdm_joblib(tqdm_object):
    class TqdmBatchCompletionCallback(
        joblib.parallel.BatchCompletionCallBack
    ):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = (
        TqdmBatchCompletionCallback
    )

    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_callback
        tqdm_object.close()

In [19]:
class ItemSimilarityComputer:
    def __init__(
        self,
        n_jobs: int = -1,
        chunk_size: int = 300,
        top_k: int = 50
    ):
        self.n_jobs = n_jobs
        self.chunk_size = chunk_size
        self.top_k = top_k


    # --------------------------------------------------
    # Step 1
    # --------------------------------------------------

    def transpose_matrix(self, user_item_matrix):

        logger.info("Transposing User-Item matrix...")

        item_user_matrix = user_item_matrix.T.tocsr()

        logger.info(
            f"Item-User shape: {item_user_matrix.shape}"
        )

        return item_user_matrix


    # --------------------------------------------------
    # Step 2
    # --------------------------------------------------

    def generate_chunks(self, n_items):

        chunks = [
            (i, min(i + self.chunk_size, n_items))
            for i in range(
                0,
                n_items,
                self.chunk_size
            )
        ]

        logger.info(
            f"Total chunks: {len(chunks)}"
        )

        return chunks


    # --------------------------------------------------
    # Step 3 — SAFE Top-K pruning
    # --------------------------------------------------

    def keep_top_k(self, similarity):

        similarity = similarity.tolil()

        for i in range(similarity.shape[0]):

            row = similarity.data[i]

            if len(row) > self.top_k:

                idx = np.argsort(row)[-self.top_k:]

                similarity.data[i] = list(
                    np.array(similarity.data[i])[idx]
                )

                similarity.rows[i] = list(
                    np.array(similarity.rows[i])[idx]
                )

        return similarity.tocsr()


    # --------------------------------------------------
    # Step 4
    # --------------------------------------------------

    def compute_chunk(
        self,
        start,
        end,
        matrix
    ):

        logger.debug(
            f"Chunk {start} → {end}"
        )

        similarity = cosine_similarity(
            matrix[start:end],
            matrix,
            dense_output=False
        )

        similarity = self.keep_top_k(similarity)

        gc.collect()

        return similarity


    # --------------------------------------------------
    # Step 5
    # --------------------------------------------------

    def compute_parallel(
        self,
        matrix,
        chunks
    ):

        logger.info(
            "Computing similarity (parallel)..."
        )

        with tqdm_joblib(
            tqdm(
                total=len(chunks),
                desc="Item Similarity",
                colour="green"
            )
        ):

            results = Parallel(
                n_jobs=self.n_jobs,
                backend="loky"
            )(
                delayed(self.compute_chunk)(
                    start,
                    end,
                    matrix
                )
                for start, end in chunks
            )

        logger.info("Parallel computation finished")

        return results


    # --------------------------------------------------
    # Step 6
    # --------------------------------------------------

    def stack_results(self, results):

        logger.info("Stacking sparse chunks...")

        similarity_matrix = vstack(results)

        del results
        gc.collect()

        return similarity_matrix


    # --------------------------------------------------
    # Main
    # --------------------------------------------------

    def compute_item_similarity(
        self,
        user_item_matrix
    ):

        logger.info(
            "Starting Item Similarity Pipeline..."
        )

        item_user_matrix = self.transpose_matrix(
            user_item_matrix
        )

        chunks = self.generate_chunks(
            item_user_matrix.shape[0]
        )

        results = self.compute_parallel(
            item_user_matrix,
            chunks
        )

        similarity_matrix = self.stack_results(
            results
        )

        logger.info(
            "Item Similarity Pipeline completed."
        )

        return similarity_matrix

In [20]:
similarity_builder = ItemSimilarityComputer(
                        n_jobs=-1,
                        chunk_size=300,
                        top_k=50,
                        )
item_similarity_matrix = similarity_builder.compute_item_similarity(user_item_matrix)

2026-04-07 04:02:52,151 | INFO | __main__ | Starting Item Similarity Pipeline...
INFO:__main__:Starting Item Similarity Pipeline...
2026-04-07 04:02:52,154 | INFO | __main__ | Transposing User-Item matrix...
INFO:__main__:Transposing User-Item matrix...
2026-04-07 04:02:52,223 | INFO | __main__ | Item-User shape: (39123, 918463)
INFO:__main__:Item-User shape: (39123, 918463)
2026-04-07 04:02:52,229 | INFO | __main__ | Total chunks: 131
INFO:__main__:Total chunks: 131
2026-04-07 04:02:52,233 | INFO | __main__ | Computing similarity (parallel)...
INFO:__main__:Computing similarity (parallel)...


Item Similarity:   0%|          | 0/131 [00:00<?, ?it/s]

2026-04-07 04:03:14,039 | INFO | __main__ | Parallel computation finished
INFO:__main__:Parallel computation finished
2026-04-07 04:03:14,041 | INFO | __main__ | Stacking sparse chunks...
INFO:__main__:Stacking sparse chunks...
2026-04-07 04:03:14,174 | INFO | __main__ | Item Similarity Pipeline completed.
INFO:__main__:Item Similarity Pipeline completed.


In [21]:
item_similarity_matrix.shape

(39123, 39123)

## Step 3. Construct Similarity Matrix

In [22]:
import gc
import numpy as np
from scipy.sparse import csr_matrix, vstack
from joblib import Parallel, delayed
from tqdm.auto import tqdm
import joblib
from contextlib import contextmanager

In [23]:
@contextmanager
def tqdm_joblib(tqdm_object):

    class TqdmBatchCompletionCallback(
        joblib.parallel.BatchCompletionCallBack
    ):
        def __call__(self, *args, **kwargs):
            tqdm_object.update(n=self.batch_size)
            return super().__call__(*args, **kwargs)

    old_callback = joblib.parallel.BatchCompletionCallBack
    joblib.parallel.BatchCompletionCallBack = (
        TqdmBatchCompletionCallback
    )

    try:
        yield tqdm_object
    finally:
        joblib.parallel.BatchCompletionCallBack = old_callback
        tqdm_object.close()

In [24]:
class SimilarityMatrixConstructor:
    def __init__(
        self,
        top_k: int = 50,
        n_jobs: int = -1,
        chunk_size: int = 500
    ):

        self.top_k = top_k
        self.n_jobs = n_jobs
        self.chunk_size = chunk_size


    # --------------------------------------------------
    # Step 1 — Generate chunks
    # --------------------------------------------------

    def generate_chunks(self, n_items):

        chunks = [
            (i, min(i + self.chunk_size, n_items))
            for i in range(
                0,
                n_items,
                self.chunk_size
            )
        ]

        logger.info(
            f"Total chunks: {len(chunks)}"
        )

        return chunks


    # --------------------------------------------------
    # Step 2 — Top-K pruning (Sparse safe)
    # --------------------------------------------------

    def prune_top_k(self, matrix):

        matrix = matrix.tolil()

        for i in range(matrix.shape[0]):

            row = matrix.data[i]

            if len(row) > self.top_k:

                idx = np.argsort(row)[-self.top_k:]

                matrix.data[i] = list(
                    np.array(matrix.data[i])[idx]
                )

                matrix.rows[i] = list(
                    np.array(matrix.rows[i])[idx]
                )

        return matrix.tocsr()


    # --------------------------------------------------
    # Step 3 — Process chunk
    # --------------------------------------------------

    def process_chunk(
        self,
        start,
        end,
        similarity_matrix
    ):

        logger.debug(
            f"Processing chunk {start} → {end}"
        )

        chunk = similarity_matrix[start:end]

        pruned = self.prune_top_k(chunk)

        gc.collect()

        return pruned


    # --------------------------------------------------
    # Step 4 — Parallel processing
    # --------------------------------------------------

    def parallel_pruning(
        self,
        similarity_matrix,
        chunks
    ):

        logger.info(
            "Running Top-K pruning (parallel)..."
        )

        with tqdm_joblib(
            tqdm(
                total=len(chunks),
                desc="Top-K Pruning",
                colour="green"
            )
        ):

            results = Parallel(
                n_jobs=self.n_jobs,
                backend="loky"
            )(
                delayed(self.process_chunk)(
                    start,
                    end,
                    similarity_matrix
                )
                for start, end in chunks
            )

        logger.info(
            "Top-K pruning completed."
        )

        return results


    # --------------------------------------------------
    # Step 5 — Stack sparse matrix
    # --------------------------------------------------

    def stack_results(self, results):

        logger.info(
            "Stacking sparse chunks..."
        )

        sparse_matrix = vstack(results)

        del results
        gc.collect()

        return sparse_matrix


    # --------------------------------------------------
    # Main Pipeline
    # --------------------------------------------------

    def construct_similarity_matrix(
        self,
        similarity_matrix
    ):

        logger.info(
            "Starting Similarity Matrix Construction..."
        )

        chunks = self.generate_chunks(
            similarity_matrix.shape[0]
        )

        results = self.parallel_pruning(
            similarity_matrix,
            chunks
        )

        sparse_similarity = self.stack_results(
            results
        )

        logger.info(
            "Similarity Matrix Construction Completed."
        )

        return sparse_similarity

In [25]:
similarity_constructor = SimilarityMatrixConstructor(
    top_k=50,
    n_jobs=-1,
    chunk_size=300
)

sparse_similarity = similarity_constructor.construct_similarity_matrix(
    item_similarity_matrix
)

2026-04-07 04:03:14,249 | INFO | __main__ | Starting Similarity Matrix Construction...
INFO:__main__:Starting Similarity Matrix Construction...
2026-04-07 04:03:14,252 | INFO | __main__ | Total chunks: 131
INFO:__main__:Total chunks: 131
2026-04-07 04:03:14,254 | INFO | __main__ | Running Top-K pruning (parallel)...
INFO:__main__:Running Top-K pruning (parallel)...


Top-K Pruning:   0%|          | 0/131 [00:00<?, ?it/s]

2026-04-07 04:03:20,483 | INFO | __main__ | Top-K pruning completed.
INFO:__main__:Top-K pruning completed.
2026-04-07 04:03:20,485 | INFO | __main__ | Stacking sparse chunks...
INFO:__main__:Stacking sparse chunks...
2026-04-07 04:03:20,645 | INFO | __main__ | Similarity Matrix Construction Completed.
INFO:__main__:Similarity Matrix Construction Completed.


In [26]:
print(sparse_similarity.shape)

(39123, 39123)


## Step 4. Predict User Preference

In [27]:
from scipy.sparse import csr_matrix, vstack
from joblib import Parallel, delayed
from tqdm import tqdm

class UserPreferencePredictor:
    """
    Memory Efficient User Preference Predictor
    - Chunk Processing
    - Parallel computation
    - tqdm progress bar
    """
    def __init__(
        self,
        n_jobs: int = -1,
        chunk_size: int = 500
    ):
        self.n_jobs = n_jobs
        self.chunk_size = chunk_size


    # --------------------------------------------------
    # Step 1 — Convert CSR
    # --------------------------------------------------

    def convert_to_csr(
        self,
        user_item_matrix,
        similarity_matrix
    ):

        logger.info("Converting matrices to CSR...")

        if not isinstance(user_item_matrix, csr_matrix):
            user_item_matrix = user_item_matrix.tocsr()

        if not isinstance(similarity_matrix, csr_matrix):
            similarity_matrix = similarity_matrix.tocsr()

        logger.debug("CSR conversion completed.")

        return user_item_matrix, similarity_matrix


    # --------------------------------------------------
    # Step 2 — Split Users
    # --------------------------------------------------

    def split_user_chunks(
        self,
        n_users
    ):

        logger.info("Splitting users into chunks...")

        chunks = [
            (i, min(i + self.chunk_size, n_users))
            for i in range(0, n_users, self.chunk_size)
        ]

        logger.debug(
            f"Total chunks: {len(chunks)}"
        )

        return chunks


    # --------------------------------------------------
    # Step 3 — Predict Chunk
    # --------------------------------------------------

    def predict_chunk(
        self,
        start,
        end,
        user_item_matrix,
        similarity_matrix
    ):

        logger.debug(
            f"Predicting users {start} → {end}"
        )

        user_chunk = user_item_matrix[start:end]

        # Sparse multiplication
        prediction_chunk = user_chunk.dot(
            similarity_matrix.T
        )

        return prediction_chunk


    # --------------------------------------------------
    # Step 4 — Parallel Processing
    # --------------------------------------------------

    def parallel_prediction(
        self,
        user_item_matrix,
        similarity_matrix,
        chunks
    ):

        logger.info(
            "Computing predictions in parallel..."
        )

        results = Parallel(
            n_jobs=self.n_jobs
        )(
            delayed(self.predict_chunk)(
                start,
                end,
                user_item_matrix,
                similarity_matrix
            )
            for start, end in tqdm(
                chunks,
                desc="Predicting Preferences"
            )
        )

        logger.info(
            "Parallel prediction completed."
        )

        return results


    # --------------------------------------------------
    # Step 5 — Combine Matrix
    # --------------------------------------------------

    def combine_prediction_chunks(
        self,
        results
    ):

        logger.info(
            "Combining prediction chunks..."
        )

        prediction_matrix = vstack(
            results,
            format="csr"
        )

        logger.debug(
            f"Prediction matrix shape: {prediction_matrix.shape}"
        )

        return prediction_matrix


    # --------------------------------------------------
    # Main Pipeline
    # --------------------------------------------------

    def predict_user_preferences(
        self,
        user_item_matrix,
        similarity_matrix
    ):

        logger.info(
            "Starting User Preference Prediction..."
        )

        # Step 1
        user_item_matrix, similarity_matrix = self.convert_to_csr(
            user_item_matrix,
            similarity_matrix
        )

        # Step 2
        chunks = self.split_user_chunks(
            user_item_matrix.shape[0]
        )

        # Step 3
        results = self.parallel_prediction(
            user_item_matrix,
            similarity_matrix,
            chunks
        )

        # Step 4
        prediction_matrix = self.combine_prediction_chunks(
            results
        )

        logger.info(
            "User Preference Prediction Completed."
        )

        return prediction_matrix

In [28]:
# Step 4
predictor = UserPreferencePredictor()

prediction_matrix = predictor.predict_user_preferences(
    user_item_matrix,
    sparse_similarity,
)

2026-04-07 04:03:20,699 | INFO | __main__ | Starting User Preference Prediction...
INFO:__main__:Starting User Preference Prediction...
2026-04-07 04:03:20,701 | INFO | __main__ | Converting matrices to CSR...
INFO:__main__:Converting matrices to CSR...
2026-04-07 04:03:20,703 | INFO | __main__ | Splitting users into chunks...
INFO:__main__:Splitting users into chunks...
2026-04-07 04:03:20,705 | INFO | __main__ | Computing predictions in parallel...
INFO:__main__:Computing predictions in parallel...
Predicting Preferences: 100%|██████████| 1837/1837 [01:08<00:00, 26.71it/s]
2026-04-07 04:04:31,026 | INFO | __main__ | Parallel prediction completed.
INFO:__main__:Parallel prediction completed.
2026-04-07 04:04:31,028 | INFO | __main__ | Combining prediction chunks...
INFO:__main__:Combining prediction chunks...
2026-04-07 04:04:34,198 | INFO | __main__ | User Preference Prediction Completed.
INFO:__main__:User Preference Prediction Completed.


In [29]:
print(prediction_matrix.shape)

(918463, 39123)


## Step 5. Generate Top-N Recommendations

In [30]:
from scipy.sparse import csr_matrix
from joblib import Parallel, delayed
from tqdm import tqdm

class TopNRecommender:
    """
    Memory Efficient Top-N Recommender
    - Sparse friendly
    - Parallel processing
    - tqdm progress bar
    """
    def __init__(
        self,
        top_n: int = 10,
        n_jobs: int = -1,
        chunk_size: int = 500
    ):
        self.top_n = top_n
        self.n_jobs = n_jobs
        self.chunk_size = chunk_size

    # --------------------------------------------------
    # Step 1 — Convert CSR
    # --------------------------------------------------

    def convert_to_csr(
        self,
        prediction_matrix,
        user_item_matrix
    ):

        logger.info("Converting matrices to CSR format...")

        if not isinstance(prediction_matrix, csr_matrix):
            prediction_matrix = prediction_matrix.tocsr()

        if not isinstance(user_item_matrix, csr_matrix):
            user_item_matrix = user_item_matrix.tocsr()

        return prediction_matrix, user_item_matrix


    # --------------------------------------------------
    # Step 2 — Split Users
    # --------------------------------------------------

    def split_user_chunks(
        self,
        n_users
    ):

        logger.info("Splitting users into chunks...")

        chunks = [
            (i, min(i + self.chunk_size, n_users))
            for i in range(0, n_users, self.chunk_size)
        ]

        logger.debug(f"Total chunks: {len(chunks)}")

        return chunks


    # --------------------------------------------------
    # Step 3 — Top-N For Single User
    # --------------------------------------------------

    def get_top_n_for_user(
        self,
        user_idx,
        prediction_matrix,
        user_item_matrix
    ):

        row = prediction_matrix.getrow(user_idx)

        # indices and values
        scores = row.data
        items = row.indices

        # Remove seen items
        seen = user_item_matrix.getrow(user_idx).indices

        mask = ~np.isin(items, seen)

        items = items[mask]
        scores = scores[mask]

        if len(scores) == 0:
            return user_idx, list(), list()

        # Top-N selection
        top_idx = np.argsort(scores)[-self.top_n:][::-1]

        return (
            user_idx,
            items[top_idx],
            scores[top_idx]
        )


    # --------------------------------------------------
    # Step 4 — Process Chunk
    # --------------------------------------------------

    def process_user_chunk(
        self,
        start,
        end,
        prediction_matrix,
        user_item_matrix
    ):

        logger.debug(f"Processing users {start} → {end}")

        results = list()

        for user_idx in range(start, end):

            result = self.get_top_n_for_user(
                user_idx,
                prediction_matrix,
                user_item_matrix
            )

            results.append(result)

        return results


    # --------------------------------------------------
    # Step 5 — Parallel Processing
    # --------------------------------------------------

    def parallel_generate(
        self,
        prediction_matrix,
        user_item_matrix,
        chunks
    ):

        logger.info("Generating Top-N in parallel...")

        results = Parallel(
            n_jobs=self.n_jobs
        )(
            delayed(self.process_user_chunk)(
                start,
                end,
                prediction_matrix,
                user_item_matrix
            )
            for start, end in tqdm(
                chunks,
                desc="Top-N Generation"
            )
        )

        logger.info("Parallel Top-N completed.")

        return results


    # --------------------------------------------------
    # Step 6 — Combine Results
    # --------------------------------------------------

    def combine_results(
        self,
        results
    ):

        logger.info("Combining recommendations...")

        recommendations = {}

        for chunk in results:
            for user_idx, items, scores in chunk:

                recommendations[user_idx] = {
                    "items": items,
                    "scores": scores
                }

        logger.debug(
            f"Total users recommended: {len(recommendations)}"
        )

        return recommendations


    # --------------------------------------------------
    # Main Pipeline
    # --------------------------------------------------

    def generate_top_n_recommendations(
        self,
        prediction_matrix,
        user_item_matrix
    ):

        logger.info(
            "Starting Top-N Recommendation Pipeline..."
        )

        # Step 1
        prediction_matrix, user_item_matrix = self.convert_to_csr(
            prediction_matrix,
            user_item_matrix
        )

        # Step 2
        chunks = self.split_user_chunks(
            prediction_matrix.shape[0]
        )

        # Step 3
        results = self.parallel_generate(
            prediction_matrix,
            user_item_matrix,
            chunks
        )

        # Step 4
        recommendations = self.combine_results(
            results
        )

        logger.info(
            "Top-N Recommendation Completed."
        )

        return recommendations

In [ ]:
topn = TopNRecommender(top_n=10)

recommendations = topn.generate_top_n_recommendations(
    prediction_matrix,
    user_item_matrix
)

2026-04-07 04:04:34,291 | INFO | __main__ | Starting Top-N Recommendation Pipeline...
INFO:__main__:Starting Top-N Recommendation Pipeline...
2026-04-07 04:04:34,294 | INFO | __main__ | Converting matrices to CSR format...
INFO:__main__:Converting matrices to CSR format...
2026-04-07 04:04:34,300 | INFO | __main__ | Splitting users into chunks...
INFO:__main__:Splitting users into chunks...
2026-04-07 04:04:34,302 | INFO | __main__ | Generating Top-N in parallel...
INFO:__main__:Generating Top-N in parallel...
Top-N Generation:  33%|███▎      | 604/1837 [00:55<02:25,  8.49it/s]

## Storing the Result

In [ ]:
import gc
import os
import duckdb
import pandas as pd
import logging

from datetime import datetime, UTC
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

In [ ]:
import gc
import os
import duckdb
import pandas as pd
import logging

from datetime import datetime, UTC
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

class RecommendationStorage:
    def __init__(
        self,
        db_path="recommendations.db",
        overwrite=False,
        user_col="user_id",
        item_col="product_id",
        dept_col="department_id",
        aisle_col="aisle_id"
    ):
        logger.debug("Initializing RecommendationStorage")

        self.db_path = db_path
        self.overwrite = overwrite

        self.user_col = user_col
        self.item_col = item_col
        self.dept_col = dept_col
        self.aisle_col = aisle_col

        self._prepare_database()


    # --------------------------------------------------
    # Prepare Database
    # --------------------------------------------------

    def _prepare_database(self):

        logger.debug("Preparing DuckDB database")

        if os.path.exists(self.db_path):

            logger.debug(f"Database exists: {self.db_path}")

            if self.overwrite:
                logger.debug("Overwrite enabled — removing existing DB")
                os.remove(self.db_path)

            else:
                logger.debug("Overwrite disabled — appending to DB")

        gc.collect()


    # --------------------------------------------------
    # Build Product Metadata
    # --------------------------------------------------

    def build_product_metadata(self, master_df):

        logger.debug("Building product metadata")

        conn = duckdb.connect()

        conn.register("master_df", master_df)

        product_meta = conn.execute(f"""
        SELECT DISTINCT
            {self.item_col} AS product_id,
            {self.dept_col} AS department_id,
            {self.aisle_col} AS aisle_id
        FROM master_df
        """).fetchdf()

        conn.close()

        logger.debug(
            f"Product metadata shape: {product_meta.shape}"
        )

        gc.collect()

        return product_meta


    # --------------------------------------------------
    # Multi-thread Convert
    # --------------------------------------------------

    def convert_to_dataframe(
        self,
        recommendations,
        n_threads: int = 12
    ):

        logger.debug(
            f"Converting recommendations using {n_threads} threads"
        )

        now = datetime.now(UTC)

        user_chunks = list(recommendations.items())

        def process_chunk(chunk):

            local_frames = []

            for user_id, recs in chunk:

                items = recs["items"]
                scores = recs["scores"]

                df_user = pd.DataFrame({
                    "user_id": user_id,
                    "product_id": items,
                    "rank": range(1, len(items) + 1),
                    "score": scores,
                    "created_at": now
                })

                local_frames.append(df_user)

            return local_frames


        chunk_size = max(1, len(user_chunks) // n_threads)

        chunks = [
            user_chunks[i:i + chunk_size]
            for i in range(0, len(user_chunks), chunk_size)
        ]

        logger.debug(
            f"Total chunks created: {len(chunks)}"
        )

        dfs = []

        with ThreadPoolExecutor(
            max_workers=n_threads
        ) as executor:

            futures = list(
                tqdm(
                    executor.map(
                        process_chunk,
                        chunks
                    ),
                    total=len(chunks),
                    desc="Multi-thread converting",
                    colour="green"
                )
            )

        for result in futures:
            dfs.extend(result)

        df = pd.concat(
            dfs,
            ignore_index=True
        )

        logger.debug(
            f"Recommendation dataframe shape: {df.shape}"
        )

        del dfs
        del futures
        del chunks
        del user_chunks

        gc.collect()

        return df

    # --------------------------------------------------
    # Joblib Parallel Convert (Faster)
    # --------------------------------------------------
    def Parallel_to_dataframe(
        self,
        recommendations,
        n_jobs: int = -1
    ):

        logger.debug(
            f"Converting recommendations using joblib n_jobs={n_jobs}"
        )

        from joblib import Parallel, delayed

        now = datetime.now(UTC)

        user_items = list(recommendations.items())

        def process_user(user_id, recs):

            items = recs["items"]
            scores = recs["scores"]

            return pd.DataFrame({
                "user_id": user_id,
                "product_id": items,
                "rank": range(1, len(items) + 1),
                "score": scores,
                "created_at": now
            })


        logger.debug(
            f"Total users to process: {len(user_items)}"
        )

        dfs = Parallel(
            n_jobs=n_jobs,
            backend="loky",
            verbose=10
        )(
            delayed(process_user)(user_id, recs)
            for user_id, recs in tqdm(
                user_items,
                desc="Joblib converting",
                colour="green"
            )
        )

        logger.debug("Concatenating dataframe...")

        df = pd.concat(
            dfs,
            ignore_index=True
        )

        logger.debug(
            f"Recommendation dataframe shape: {df.shape}"
        )

        del dfs
        del user_items

        gc.collect()

        return df

    # --------------------------------------------------
    # Store DuckDB
    # --------------------------------------------------

    def store_duckdb(
        self,
        recommendation_df,
        product_meta
    ):

        logger.debug("Storing into DuckDB")

        conn = duckdb.connect(
            self.db_path
        )

        conn.register(
            "recommendations_df",
            recommendation_df
        )

        conn.register(
            "product_meta",
            product_meta
        )

        conn.execute("""
        CREATE TABLE IF NOT EXISTS recommendations (
            user_id INTEGER,
            product_id INTEGER,
            department_id INTEGER,
            aisle_id INTEGER,
            rank INTEGER,
            score FLOAT,
            created_at TIMESTAMP
        )
        """)

        logger.debug("Inserting recommendations")

        conn.execute("""
        INSERT INTO recommendations
        SELECT
            r.user_id,
            r.product_id,
            p.department_id,
            p.aisle_id,
            r.rank,
            r.score,
            r.created_at
        FROM recommendations_df r
        LEFT JOIN product_meta p
        ON r.product_id = p.product_id
        """)

        logger.debug("Creating index")

        conn.execute("""
        CREATE INDEX IF NOT EXISTS idx_user
        ON recommendations(user_id)
        """)

        conn.close()

        gc.collect()


    # --------------------------------------------------
    # Callable Pipeline
    # --------------------------------------------------

    def __call__(
        self,
        recommendations,
        master_df,
        multithread : bool = False,
        n_threads:int = 12,
    ):

        logger.debug("Starting Recommendation Storage Pipeline")

        product_meta = self.build_product_metadata(
            master_df
        )

        gc.collect()

        if multithread:
            recommendation_df = self.convert_to_dataframe(
                recommendations,
                n_threads=n_threads
            )
        else:
            recommendation_df = self.Parallel_to_dataframe(recommendations)

        gc.collect()

        self.store_duckdb(
            recommendation_df,
            product_meta
        )

        logger.debug("Recommendation Storage Completed")

        gc.collect()

        return recommendation_df

In [ ]:
storage = RecommendationStorage(
    db_path="Item_Filtering_recommendations.db",
    overwrite=True
)

recommendation_df = storage(
    recommendations,
    masterdata2,
    n_threads=12,
)